# Project 10 -- Calgary Water Quality Anomaly Detection
## Exploratory Data Analysis

This notebook performs initial exploration of the Watershed Water Quality dataset from the Calgary Open Data portal. We will:

1. Load and inspect the raw data
2. Examine parameter distributions
3. Explore time series patterns
4. Identify potential anomalies visually

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root so we can import src modules
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    fetch_water_quality_data,
    preprocess,
    pivot_parameters,
    add_rolling_statistics,
    add_rate_of_change,
    add_zscore_features,
    load_and_prepare,
    KEY_PARAMETERS,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
print("Imports complete.")

## 1. Load and Inspect the Raw Data

In [ ]:
# Fetch raw data (cached locally after first download)
raw_df = fetch_water_quality_data(limit=50_000)
print(f"Raw data shape: {raw_df.shape}")
raw_df.head()

In [ ]:
# Data types and missing values
raw_df.info()

In [ ]:
# Preprocess: parse dates, convert numeric_result, extract year/month
long_df = preprocess(raw_df)
print(f"Preprocessed shape: {long_df.shape}")
print(f"Date range: {long_df['sample_date'].min()} to {long_df['sample_date'].max()}")
print(f"Unique sites: {long_df['sample_site'].nunique()}")
print(f"Unique parameters: {long_df['parameter'].nunique()}")
long_df.head()

In [ ]:
# List all monitoring sites
print("Monitoring sites:")
for site in sorted(long_df["sample_site"].dropna().unique()):
    count = len(long_df[long_df["sample_site"] == site])
    print(f"  {site}: {count:,} records")

In [ ]:
# List all parameters and their frequency
param_counts = long_df["parameter"].value_counts()
print(f"Total unique parameters: {len(param_counts)}")
param_counts.head(20)

## 2. Parameter Distributions

In [ ]:
# Distribution of key parameters (histograms)
available_key = [p for p in KEY_PARAMETERS if p in long_df["parameter"].values]
if not available_key:
    available_key = list(long_df["parameter"].value_counts().head(5).index)

for param in available_key:
    subset = long_df[long_df["parameter"] == param].dropna(subset=["numeric_result"])
    if subset.empty:
        continue
    fig = px.histogram(
        subset, x="numeric_result", nbins=50,
        title=f"Distribution of {param} (n={len(subset):,})",
        labels={"numeric_result": param},
        marginal="box",
    )
    fig.update_layout(template="plotly_white", height=350)
    fig.show()

In [ ]:
# Summary statistics per parameter
param_stats = (
    long_df.dropna(subset=["numeric_result"])
    .groupby("parameter")["numeric_result"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .round(3)
    .sort_values("count", ascending=False)
)
param_stats.head(20)

## 3. Time Series Exploration

In [ ]:
# Time series of key parameters at the most-sampled site
top_site = long_df["sample_site"].value_counts().index[0]
print(f"Most-sampled site: {top_site}")

site_data = long_df[
    (long_df["sample_site"] == top_site) & (long_df["parameter"].isin(available_key))
].dropna(subset=["numeric_result"])

fig_ts = px.line(
    site_data, x="sample_date", y="numeric_result",
    color="parameter", facet_row="parameter",
    title=f"Key Parameters Over Time -- {top_site}",
    labels={"sample_date": "Date", "numeric_result": "Value"},
)
fig_ts.update_layout(height=250 * len(available_key), template="plotly_white")
fig_ts.update_yaxes(matches=None)
fig_ts.show()

In [ ]:
# Monthly aggregation to see seasonal trends
monthly = (
    long_df[long_df["parameter"].isin(available_key)]
    .dropna(subset=["numeric_result"])
    .groupby(["parameter", "month"])["numeric_result"]
    .agg(["mean", "std"])
    .reset_index()
)

fig_month = px.line(
    monthly, x="month", y="mean", color="parameter",
    error_y="std",
    title="Monthly Mean of Key Parameters (with std dev bands)",
    labels={"month": "Month", "mean": "Mean Value"},
)
fig_month.update_layout(template="plotly_white", height=450)
fig_month.show()

In [ ]:
# Yearly sample counts -- data coverage over time
yearly_counts = long_df.groupby("year").size().reset_index(name="count")
fig_yr = px.bar(
    yearly_counts, x="year", y="count",
    title="Number of Samples per Year",
    labels={"year": "Year", "count": "Sample Count"},
)
fig_yr.update_layout(template="plotly_white", height=350)
fig_yr.show()

## 4. Initial Anomaly Identification

Before applying formal anomaly detection models, we use simple statistical rules to flag potential outliers.

In [ ]:
# Z-score based outlier identification per parameter
def flag_outliers(group, n_std=3):
    """Flag rows where numeric_result exceeds mean +/- n_std."""
    mean = group["numeric_result"].mean()
    std = group["numeric_result"].std()
    if std and std > 0:
        group["zscore"] = (group["numeric_result"] - mean) / std
        group["is_outlier"] = group["zscore"].abs() > n_std
    else:
        group["zscore"] = 0.0
        group["is_outlier"] = False
    return group

flagged = (
    long_df.dropna(subset=["numeric_result"])
    .groupby("parameter", group_keys=False)
    .apply(flag_outliers)
)

outlier_summary = (
    flagged.groupby("parameter")["is_outlier"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "outliers", "count": "total"})
    .assign(pct=lambda d: (d["outliers"] / d["total"] * 100).round(2))
    .sort_values("outliers", ascending=False)
)

print("Outlier counts per parameter (3-sigma rule):")
outlier_summary.head(15)

In [ ]:
# Visualise outliers on a scatter plot for a key parameter
vis_param = available_key[0] if available_key else flagged["parameter"].value_counts().index[0]
vis_data = flagged[flagged["parameter"] == vis_param].copy()

fig_out = go.Figure()
normal = vis_data[~vis_data["is_outlier"]]
outliers = vis_data[vis_data["is_outlier"]]

fig_out.add_trace(go.Scatter(
    x=normal["sample_date"], y=normal["numeric_result"],
    mode="markers", name="Normal",
    marker=dict(color="steelblue", size=4, opacity=0.5),
))
fig_out.add_trace(go.Scatter(
    x=outliers["sample_date"], y=outliers["numeric_result"],
    mode="markers", name="Outlier (3-sigma)",
    marker=dict(color="red", size=8, symbol="x"),
))
fig_out.update_layout(
    title=f"Statistical Outliers in {vis_param}",
    xaxis_title="Date", yaxis_title=vis_param,
    template="plotly_white", height=400,
)
fig_out.show()

In [ ]:
# Pivot and prepare wide-form data for modelling
long_df_full, wide_df = load_and_prepare()
print(f"Wide-form shape: {wide_df.shape}")
print(f"Columns: {list(wide_df.columns[:20])} ...")
wide_df.head()

In [ ]:
# Quick correlation check among raw parameter columns
exclude = {"sample_site", "sample_date", "year", "month"}
param_cols = [
    c for c in wide_df.columns
    if c not in exclude
    and "_roll_" not in c
    and "_roc" not in c
    and "_zscore" not in c
    and wide_df[c].dtype in ("float64", "int64")
]

if len(param_cols) >= 2:
    corr = wide_df[param_cols].corr()
    fig_corr = px.imshow(
        corr, text_auto=".2f", color_continuous_scale="RdBu_r",
        title="Parameter Correlation Matrix", aspect="auto",
    )
    fig_corr.update_layout(height=500, template="plotly_white")
    fig_corr.show()
else:
    print("Not enough numeric parameters for correlation matrix.")

## Summary

Key findings from this exploratory analysis:

- The dataset contains measurements from multiple monitoring sites across Calgary's watershed.
- Several parameters show clear seasonal patterns (e.g., temperature peaks in summer).
- Simple statistical thresholds (3-sigma) already identify a small number of outliers per parameter.
- The wide-form, feature-engineered dataset is ready for formal multi-method anomaly detection in `src/model.py`.

Next steps: apply the ensemble anomaly detector (Isolation Forest + LOF + Statistical + Z-Score) and evaluate results in the Streamlit dashboard.